# DSA 4060 — Week 9 Guided Lab
## From implicit events to ranked recommendations with confidence-weighted ALS

**Steps in this notebook**
1. Set up
2. Create an event log
3. Turn events into interaction strength
4. Preference and confidence
5. Build the sparse user–item matrix
6. Train an ALS model
7. Generate Top-5 recommendations
8. One-variable experiment
9. Evaluation (time-aware split)
10. Exit ticket

## 1. Set up
Select your kernel after installing the required packages on the terminal, then run the install cell. 

In [2]:
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from implicit.als import AlternatingLeastSquares

np.random.seed(4060)

**Discuss:** 
• Which signals in a streaming, retail, or learning platform are explicit? Which are implicit?
• Does “no interaction” mean dislike, lack of awareness, or simply no opportunity?

Explicit feedback is when users directly rate or like an item, while implicit feedback is based on their behaviour, such as plays, skips, or saves; however, no interaction does not necessarily mean dislike because the user may never have seen or had the opportunity to interact with the item, so it should be treated as unknown.

## 2. Create a small event log
Each row is one observed event, **not** a rating.

In [3]:
users = [f"U{i:02d}" for i in range(1, 13)]
items = [f"Track_{i:02d}" for i in range(1, 19)]
event_types = ["play", "repeat_play", "save"]

rows = []
for user in users:
    preferred = np.random.choice(items, size=6, replace=False)
    for item in preferred:
        n = np.random.randint(1, 5)
        for _ in range(n):
            rows.append({
                "user_id": user,
                "item_id": item,
                "event_type": np.random.choice(event_types, p=[0.60, 0.25, 0.15])
            })

events = pd.DataFrame(rows)
print(events.shape)
events.head()

(194, 3)


,user_id,item_id,event_type
0,U01,Track_02,repeat_play
1,U01,Track_02,repeat_play
2,U01,Track_02,play
3,U01,Track_02,save
4,U01,Track_06,repeat_play


In [4]:
print(events['event_type'].value_counts())
events.groupby('user_id').size().describe()

event_type
play           120
repeat_play     44
save            30
Name: count, dtype: int64


count    12.000000
mean     16.166667
std       3.511885
min      12.000000
25%      13.750000
50%      15.000000
75%      17.750000
max      23.000000
dtype: float64

### Checkpoint 1 — What do the rows mean?
- What can we infer from a repeat play or a save?
- What can we **not** infer?
- Which biases might appear in this synthetic log, and in a real platform log?

A repeat play or save suggests that a user has some interest in a track, but we cannot know exactly why they interacted with it or assume they genuinely like it. The data may also contain biases, such as certain tracks being exposed more often, users having different listening habits, or synthetic events not reflecting real human behaviour.

## 3. Turn events into interaction strength
These weights are modeling choices, not universal truths.

In [5]:
event_weight = {
    "play": 1.0,
    "repeat_play": 3.0,
    "save": 5.0
}

events['weight'] = events['event_type'].map(event_weight)

interactions = (events
    .groupby(['user_id', 'item_id'], as_index=False)['weight']
    .sum()
    .rename(columns={'weight': 'strength'}))

interactions.sort_values('strength', ascending=False).head(10)

,user_id,item_id,strength
1,U01,Track_02,12.0
71,U12,Track_12,12.0
61,U11,Track_03,12.0
69,U12,Track_09,12.0
14,U03,Track_11,11.0
70,U12,Track_10,11.0
8,U02,Track_10,10.0
11,U02,Track_17,10.0
67,U12,Track_05,10.0
54,U10,Track_06,10.0


### Checkpoint 2 — Challenge the weights
- Should one save always be worth five plays?
- How could accidental clicks, autoplay, or repeated exposure distort strength?
- What different weights would your group test first, and why?

The weights assign different levels of importance to user actions, assuming that a save shows stronger interest than a repeat play or a normal play. However, one save is not always worth five plays, because accidental clicks, autoplay and repeated exposure can distort the results. We should test different weights to see which combination best represents genuine user interest.

## 4. Preference and confidence
- **Preference** = was a positive behaviour observed? (0 or 1)
- **Confidence** = how strongly should the model trust it? (`1 + alpha * strength`)

In [6]:
alpha = 20

interactions['preference'] = (interactions['strength'] > 0).astype(int)
interactions['confidence'] = 1 + alpha * interactions['strength']

interactions[['user_id', 'item_id', 'strength', 'preference', 'confidence']].head()

,user_id,item_id,strength,preference,confidence
0,U01,Track_01,2.0,1,41.0
1,U01,Track_02,12.0,1,241.0
2,U01,Track_06,6.0,1,121.0
3,U01,Track_08,3.0,1,61.0
4,U01,Track_11,8.0,1,161.0


### Checkpoint 3 — Compare two cases
- How do preference and confidence differ for strength = 1 versus strength = 12?
- Why should unobserved pairs have low confidence rather than be treated as dislikes?
- What happens if alpha is extremely large?

For strength = 1, preference is 1 and confidence is 21, while for strength = 12, preference is also 1 but confidence is 241, so the preference is the same but the model trusts the stronger case much more. Unobserved pairs should have low confidence rather than be treated as dislikes, because the user may never have seen the item or had the opportunity to interact with it. If alpha is extremely large, the observed interactions dominate the model, so even a single event gets too much influence and the model may overfit.

## 5. Build the sparse user–item matrix
Note: the matrix stores **strength**. The model applies `alpha` itself.

In [7]:
user_ids = sorted(interactions['user_id'].unique())
item_ids = sorted(interactions['item_id'].unique())

user_to_idx = {u: i for i, u in enumerate(user_ids)}
item_to_idx = {it: i for i, it in enumerate(item_ids)}
idx_to_item = {i: it for it, i in item_to_idx.items()}

row = interactions['user_id'].map(user_to_idx)
col = interactions['item_id'].map(item_to_idx)
data = interactions['strength'].astype(float)

user_item = csr_matrix(
    (data, (row, col)),
    shape=(len(user_ids), len(item_ids))
)

print('shape:', user_item.shape)
print('observed interactions:', user_item.nnz)
print('density:', user_item.nnz / (user_item.shape[0] * user_item.shape[1]))

shape: (12, 18)
observed interactions: 72
density: 0.3333333333333333


### Checkpoint 4 — Read the matrix
- What does one row represent? What does one column represent?
- Why is a sparse matrix better than a dense table for a large catalog?
- What does density tell us about the recommendation problem?

One row represents one user, one column represents one track, and each cell stores the interaction strength between them. A sparse matrix is better for a large catalog because it stores only the interactions that exist instead of every empty cell, which saves memory and time. The density here is 0.33, which is quite high, but real platforms are much sparser, so there is less data per user and recommending becomes harder.


## 6. Train an ALS model

In [8]:
model = AlternatingLeastSquares(
    factors=16,
    regularization=0.10,
    alpha=alpha,
    iterations=25,
    random_state=4060
)

model.fit(user_item)
print(model.user_factors.shape, model.item_factors.shape)

c:\Users\Ayan\AppData\Local\Programs\Python\Python312\Lib\site-packages\implicit\cpu\als.py:96: RuntimeWarning: OpenBLAS is configured to use 12 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()


  0%|          | 0/25 [00:00<?, ?it/s]

(12, 16) (18, 16)


### Checkpoint 5 — Connect code to concepts
- What does `factors=16` control?
- Why might regularization help?
- Why is a random seed useful during a class demonstration?

factors=16 controls how many hidden features are used to describe each user and each item, so more factors capture more detail but can overfit. Regularization helps because it stops the model from relying too heavily on a few interactions, so it generalizes better to new items. A random seed is useful in a class demonstration because it makes the random starting values the same every time, so everyone gets the same results.

## 7. Generate Top-5 recommendations

In [9]:
target_user = 'U01'
u = user_to_idx[target_user]

item_idx, scores = model.recommend(
    u,
    user_item[u],
    N=5,
    filter_already_liked_items=True
)

recommendations = pd.DataFrame({
    'item_id': [idx_to_item[i] for i in item_idx],
    'score': scores
})

seen = interactions.loc[
    interactions['user_id'].eq(target_user),
    ['item_id', 'strength']
].sort_values('strength', ascending=False)

print('Observed history')
display(seen.head(10))
print('Recommended next')
display(recommendations)

Observed history


,item_id,strength
1,Track_02,12.0
4,Track_11,8.0
5,Track_13,6.0
2,Track_06,6.0
3,Track_08,3.0
0,Track_01,2.0


Recommended next


,item_id,score
0,Track_05,0.186877
1,Track_04,0.125515
2,Track_09,0.093726
3,Track_07,0.069388
4,Track_12,0.063325


### Checkpoint 6 — Critique the ranking
- Do the recommended items look plausible given the user's history?
- Why do model scores not behave like probabilities?
- What constraints might require reranking (diversity, freshness, availability, safety)?

The recommended items for U01 (Track_05, Track_04, Track_09, Track_07 and Track_12) look reasonable, as they are tracks the user has not heard and come from similar users’ tastes. However, the data is random, so the plausibility is limited. Model scores do not behave like probabilities because they are dot products of user and item factors, so they are not limited to 0–1 and are only useful for ranking. Reranking may be needed for diversity, freshness, availability and safety.

## 8. One-variable experiment
Change **only one** setting, retrain, and compare with the baseline.

| Experiment | Values |
|---|---|
| Confidence scale | alpha = 5 vs 40 |
| Latent dimension | factors = 4 vs 32 |
| Regularization | 0.01 vs 1.0 |

In [10]:
# Change ONE value below, then run this cell.
exp_model = AlternatingLeastSquares(
    factors=16,            # try 4 or 32
    regularization=0.10,   # try 0.01 or 1.0
    alpha=alpha,           # try 5 or 40
    iterations=25,
    random_state=4060
)
exp_model.fit(user_item)

exp_idx, exp_scores = exp_model.recommend(u, user_item[u], N=5, filter_already_liked_items=True)

comparison = pd.DataFrame({
    'baseline_item': recommendations['item_id'],
    'experiment_item': [idx_to_item[i] for i in exp_idx]
})
comparison

  0%|          | 0/25 [00:00<?, ?it/s]

,baseline_item,experiment_item
0,Track_05,Track_05
1,Track_04,Track_04
2,Track_09,Track_09
3,Track_07,Track_07
4,Track_12,Track_12


## 9. Evaluation: design first, then calculate
The lab data has no timestamps, so we **simulate** time by treating the order of rows in `events` as time order. In real logs, sort by a real timestamp.

Plan:
1. Sort each user's events by time.
2. Train on earlier interactions.
3. Hold out later interactions.
4. Recommend Top-K without already-seen items.
5. Compute Precision@K and Recall@K, and compare with a popularity baseline.

In [11]:
# Simulated time split: last distinct item per user is the test item
events_ordered = events.reset_index(drop=True).copy()
events_ordered['t'] = events_ordered.index

test_rows = []
train_events = []
for user, g in events_ordered.groupby('user_id'):
    g = g.sort_values('t')
    last_item = g.iloc[-1]['item_id']
    test_rows.append((user, last_item))
    train_events.append(g[g['item_id'] != last_item])

train_df = pd.concat(train_events)
train_int = (train_df.groupby(['user_id', 'item_id'], as_index=False)['weight']
             .sum().rename(columns={'weight': 'strength'}))

train_mat = csr_matrix(
    (train_int['strength'].astype(float),
     (train_int['user_id'].map(user_to_idx), train_int['item_id'].map(item_to_idx))),
    shape=user_item.shape
)

eval_model = AlternatingLeastSquares(factors=16, regularization=0.10, alpha=alpha,
                                     iterations=25, random_state=4060)
eval_model.fit(train_mat)

K = 5
# Popularity baseline: most-interacted items in training
popularity = np.asarray(train_mat.sum(axis=0)).ravel()

als_hits = pop_hits = 0
for user, test_item in test_rows:
    ui = user_to_idx[user]
    ti = item_to_idx[test_item]

    rec_idx, _ = eval_model.recommend(ui, train_mat[ui], N=K, filter_already_liked_items=True)
    als_hits += int(ti in rec_idx)

    seen_idx = set(train_mat[ui].indices)
    pop_rank = [i for i in np.argsort(-popularity) if i not in seen_idx][:K]
    pop_hits += int(ti in pop_rank)

n = len(test_rows)
print(f"ALS        Recall@{K} (Precision@{K} = Recall@{K} / {K}): {als_hits / n:.3f}")
print(f"Popularity Recall@{K}: {pop_hits / n:.3f}")

  0%|          | 0/25 [00:00<?, ?it/s]

ALS        Recall@5 (Precision@5 = Recall@5 / 5): 0.667
Popularity Recall@5: 0.583


> With only 12 users, these numbers are very noisy. Use them to learn the **method**, not to judge the model.

### Checkpoint 7 — Evaluation discussion
- Why is a random row split risky for event logs?
- When would Recall@K matter more than Precision@K?
- What offline result would still not prove that users benefit?

A random row split is risky for event logs because the model can train on later events and be tested on earlier ones, which leaks future information and makes the results look better than they are. Recall@K matters more than Precision@K when we want to find as many of the user’s relevant items as possible, such as in discovery or when generating candidates. An offline result still would not prove that users benefit, because it only shows that the model matched past behaviour, not that users were more satisfied, so a live A/B test is needed.

## 10. Exit ticket 
Answer all four:
1. Name one assumption made when converting events into weights.
2. Explain preference versus confidence using one concrete example.
3. Describe one change that altered, or did not alter, your Top-5 ranking.
4. Identify one limitation of this lab model and one next improvement.

One assumption I made was that a save is worth five plays and a repeat play is worth three, which assumes stronger actions always mean stronger interest. Preference only shows whether a user showed interest (1 or 0), while confidence shows how much the model should trust it. For example, U01 played Track_01 (strength 2) and replayed or saved Track_02 (strength 12); both have preference 1, but their confidence is 41 and 241. In my experiment, changing one setting did not alter the Top-5 ranking, which stayed Track_05, Track_04, Track_09, Track_07 and Track_12, probably because the dataset is so small. One limitation of this lab model is that the data is synthetic and has only 12 users, so the results are noisy and may not reflect how real users behave. A next improvement would be to use a real event log with timestamps and report NDCG@5 alongside Recall@5, so the evaluation is more realistic.